# Box-Cox para segmentación de grietas — **Protocolo 2 (generalización cruzada)**

A diferencia del Protocolo 1 (entrenar y testear en la *misma* imagen), aquí **train y test son
imágenes disjuntas**: mide generalización, que es el protocolo estándar de segmentación.

### Diseño
Para cada set (Vertical, Horizontal, Left, Big) se hace **validación cruzada 5-fold sobre las 50
imágenes**:
- En cada fold, se entrena cada clasificador con una **muestra estratificada de píxeles de las
  imágenes de entrenamiento** (≈40 imágenes), y
- se predice **cada imagen de test** (las ≈10 restantes) completa.
- Cada imagen es test exactamente una vez ⇒ **50 métricas por imagen** por (set, método, condición),
  sin que ningún modelo vea la imagen que evalúa.

Así se conserva el reporte "media ± std sobre 50 imágenes / set" pero con train/test disjuntos.

> Box-Cox se aplica **por imagen** (cada imagen con su propio λ y stretching), tanto en train como
> en test — es un prefiltrado, no depende del conjunto.

**Métricas:** IoU, Dice/F1, Precision, Recall, Accuracy (clase grieta). Accuracy es engañosa
(grieta ≈1% de píxeles). **Estadística:** media±std, test pareado (permutación + Wilcoxon) e IC95%
bootstrap para Δ (con − sin Box-Cox), pareando por imagen.


## 0. Dependencias (ejecutar una vez)

In [ ]:
# %pip install -q numpy scipy scikit-learn lightgbm opencv-python-headless pandas

## 1. Configuración y prefiltrado Box-Cox

In [ ]:
import os, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from scipy import stats
warnings.filterwarnings("ignore")

ROOT = Path.cwd() if (Path.cwd() / "rgb").exists() else Path("/Users/sebastianvidalsalado/Desktop/Crack_Seg")
RGB_DIR = ROOT / "rgb"
SETS = {"Vertical": "Vertical", "Horizontal": "Horizontal", "Left": "Left", "Big": "Big"}
SIZE = 256
SEED = 42
N_SPLITS = 5                 # folds de la CV cruzada
CAP_PER_CLASS_PER_IMG = 100  # píxeles por clase por imagen de entrenamiento
METRICS = ["IoU", "Dice", "Precision", "Recall", "Accuracy"]
SET_ORDER = ["Vertical", "Horizontal", "Left", "Big"]
METHOD_ORDER = ["SVM", "LightGBM", "KNN", "LDA", "QDA"]
print("ROOT =", ROOT, "| existe rgb:", RGB_DIR.exists())

def to_gray(rgb):
    rgb = rgb.astype(np.float64)
    return 0.299*rgb[:,:,0] + 0.587*rgb[:,:,1] + 0.114*rgb[:,:,2]

def boxcox_stretch(gray):
    x = gray.flatten().astype(np.float64)
    rng = x.max() - x.min()
    if rng == 0:
        return np.zeros_like(gray), np.nan
    xn = (x - x.min())/rng + 1e-5
    bc, lam = stats.boxcox(xn)
    brng = bc.max() - bc.min()
    bc = (bc - bc.min())/brng if brng > 0 else np.zeros_like(bc)
    return bc.reshape(gray.shape), float(lam)

def load_pair(rgb_path, mask_path, size=SIZE):
    rgb = cv2.cvtColor(cv2.imread(str(rgb_path)), cv2.COLOR_BGR2RGB)
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    rgb = cv2.resize(rgb, (size, size), interpolation=cv2.INTER_AREA)
    mask = cv2.resize(mask, (size, size), interpolation=cv2.INTER_NEAREST)
    return rgb, (mask > 127).astype(np.uint8)

def make_feature(rgb, use_boxcox):
    gray = to_gray(rgb)
    if use_boxcox:
        return boxcox_stretch(gray)
    return gray/255.0, np.nan

## 2. Modelos y métricas

In [ ]:
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
import lightgbm as lgb

def build_models():
    return {
        "SVM": SVC(kernel="rbf", C=1.0, gamma="scale", cache_size=500),
        "LightGBM": lgb.LGBMClassifier(n_estimators=100, num_leaves=31, verbose=-1),
        "KNN": KNeighborsClassifier(n_neighbors=5),
        "LDA": LinearDiscriminantAnalysis(),
        "QDA": QuadraticDiscriminantAnalysis(reg_param=0.01),
    }

def metrics(y_true, y_pred):
    yt, yp = y_true.astype(bool), y_pred.astype(bool)
    tp = np.sum(yt & yp); fp = np.sum(~yt & yp); fn = np.sum(yt & ~yp); tn = np.sum(~yt & ~yp)
    union = tp + fp + fn
    return dict(
        IoU=tp/union if union > 0 else np.nan,
        Dice=2*tp/(2*tp+fp+fn) if (2*tp+fp+fn) > 0 else np.nan,
        Precision=tp/(tp+fp) if (tp+fp) > 0 else 0.0,
        Recall=tp/(tp+fn) if (tp+fn) > 0 else 0.0,
        Accuracy=(tp+tn)/(tp+tn+fp+fn),
    )

## 3. Pipeline cruzado (5-fold sobre las 50 imágenes de cada set)

In [ ]:
from sklearn.model_selection import KFold

def sample_train_pixels(files, folder, use_bc, rng):
    Xs, ys = [], []
    for f in files:
        rgb, mask = load_pair(RGB_DIR/f, ROOT/folder/f)
        y = mask.reshape(-1)
        feat, _ = make_feature(rgb, use_bc)
        x = feat.reshape(-1)
        i0, i1 = np.where(y == 0)[0], np.where(y == 1)[0]
        t0 = rng.choice(i0, min(CAP_PER_CLASS_PER_IMG, len(i0)), replace=False)
        t1 = rng.choice(i1, min(CAP_PER_CLASS_PER_IMG, len(i1)), replace=False)
        idx = np.concatenate([t0, t1])
        Xs.append(x[idx]); ys.append(y[idx])
    return np.concatenate(Xs).reshape(-1, 1), np.concatenate(ys)

def run_crossimage(quick=False):
    rng = np.random.default_rng(SEED)
    rows = []
    for set_name, folder in SETS.items():
        files = sorted(f for f in os.listdir(ROOT/folder)
                       if f.lower().endswith(".jpg") and (RGB_DIR/f).exists())
        if quick:
            files = files[:10]
        files = np.array(files)
        kf = KFold(n_splits=2 if quick else N_SPLITS, shuffle=True, random_state=SEED)
        for fold, (tr_i, te_i) in enumerate(kf.split(files)):
            tr_files, te_files = files[tr_i], files[te_i]
            for cond in ("noBC", "BC"):
                use_bc = cond == "BC"
                Xtr, ytr = sample_train_pixels(tr_files, folder, use_bc, rng)
                cache = {}
                for f in te_files:
                    rgb, mask = load_pair(RGB_DIR/f, ROOT/folder/f)
                    feat, lam = make_feature(rgb, use_bc)
                    cache[f] = (feat.reshape(-1, 1), mask.reshape(-1), lam)
                for mname, model in build_models().items():
                    try:
                        model.fit(Xtr, ytr)
                    except Exception:
                        continue
                    for f, (Xte, yte, lam) in cache.items():
                        rows.append(dict(set=set_name, image=f, method=mname, condition=cond,
                                         lam=lam, fold=fold, **metrics(yte, model.predict(Xte))))
            print(f"[{set_name}] fold {fold+1} listo", end="\r")
    return pd.DataFrame(rows)

Corre el experimento. `quick=True` usa 10 img/set y 2 folds (~1.5 min); `quick=False` usa las 50 y 5 folds (~10–15 min).

In [ ]:
df = run_crossimage(quick=True)      # cambia a quick=False para la corrida completa
df.to_csv(ROOT / "results_ml_crossimg.csv", index=False)
print("\nfilas:", len(df))
df.groupby(["method", "condition"])[METRICS].mean().round(3)

## 4. Agregación, test pareado y bootstrap (igual que Protocolo 1)

In [ ]:
N_BOOT = 10000

def paired_perm_p(diff, n_perm=10000, rng=None):
    diff = diff[~np.isnan(diff)]
    if len(diff) == 0 or np.allclose(diff, 0):
        return np.nan
    obs = diff.mean(); rng = rng or np.random.default_rng(SEED)
    perm = (rng.choice([-1, 1], size=(n_perm, len(diff))) * diff).mean(axis=1)
    return float((np.abs(perm) >= abs(obs) - 1e-12).mean())

def boot_ci(x, n_boot=N_BOOT, rng=None, alpha=0.05):
    x = x[~np.isnan(x)]
    if len(x) == 0:
        return (np.nan, np.nan)
    rng = rng or np.random.default_rng(SEED)
    means = x[rng.integers(0, len(x), size=(n_boot, len(x)))].mean(axis=1)
    return tuple(np.percentile(means, [100*alpha/2, 100*(1-alpha/2)]))

def wilcoxon_p(bc, nobc):
    d = (bc - nobc); d = d[~np.isnan(d)]
    if len(d) == 0 or np.allclose(d, 0):
        return np.nan
    try:
        return float(stats.wilcoxon(bc, nobc).pvalue)
    except Exception:
        return np.nan

def build_summary(df, method_order):
    rng = np.random.default_rng(SEED); rows = []
    for s in SET_ORDER:
        for m in method_order:
            sub = df[(df["set"] == s) & (df["method"] == m)]
            if sub.empty:
                continue
            wide = sub.pivot_table(index="image", columns="condition", values=METRICS)
            for metric in METRICS:
                nobc = wide[(metric, "noBC")].to_numpy(); bc = wide[(metric, "BC")].to_numpy()
                diff = bc - nobc; lo, hi = boot_ci(diff, rng=rng)
                rows.append(dict(set=s, method=m, metric=metric, n=len(nobc),
                    noBC_mean=np.nanmean(nobc)*100, noBC_std=np.nanstd(nobc, ddof=1)*100,
                    BC_mean=np.nanmean(bc)*100, BC_std=np.nanstd(bc, ddof=1)*100,
                    diff=np.nanmean(diff)*100, ci_low=lo*100, ci_high=hi*100,
                    p_perm=paired_perm_p(diff, rng=rng), p_wilcoxon=wilcoxon_p(bc, nobc)))
    return pd.DataFrame(rows)

summary = build_summary(df, METHOD_ORDER)
summary.to_csv(ROOT / "summary_crossimg.csv", index=False)
summary.round(3).head(20)

### Tabla por set (media ± std, %) — `*` p<0.05, `**` p<0.01, `***` p<0.001 (permutación)

In [ ]:
def stars(p):
    return "" if pd.isna(p) else ("***" if p < .001 else "**" if p < .01 else "*" if p < .05 else "")

def render_set(summary, s, metric="IoU"):
    sub = summary[(summary["set"] == s) & (summary["metric"] == metric)]
    out = []
    for _, r in sub.iterrows():
        out.append({"Método": r.method, "Sin Box-Cox": f"{r.noBC_mean:.1f} ± {r.noBC_std:.1f}",
                    "Con Box-Cox": f"{r.BC_mean:.1f} ± {r.BC_std:.1f}",
                    "Δ (pp)": f"{r['diff']:+.1f}{stars(r.p_perm)}",
                    "IC95% Δ": f"[{r.ci_low:+.1f}, {r.ci_high:+.1f}]", "p (perm)": f"{r.p_perm:.3f}"})
    return pd.DataFrame(out)

for s in SET_ORDER:
    print(f"\n===== Set: {s} | métrica: IoU (cross-image) =====")
    display(render_set(summary, s, "IoU"))

### Exportar tabla Markdown (todas las métricas)

In [ ]:
def to_markdown(summary, method_order):
    lines = ["# Protocolo 2 (cross-image): Box-Cox vs sin Box-Cox — grietas (media±std %, 50 img/set)", ""]
    for s in SET_ORDER:
        lines += [f"## Set: {s}", "",
                  "| Método | Métrica | Sin Box-Cox | Con Box-Cox | Δ (pp) | IC95% Δ | p(perm) | p(Wilcoxon) |",
                  "|---|---|---|---|---|---|---|---|"]
        for m in method_order:
            for metric in METRICS:
                r = summary[(summary["set"]==s)&(summary["method"]==m)&(summary["metric"]==metric)]
                if r.empty: continue
                r = r.iloc[0]
                lines.append(f"| {m} | {metric} | {r.noBC_mean:.1f} ± {r.noBC_std:.1f} | "
                             f"{r.BC_mean:.1f} ± {r.BC_std:.1f} | {r['diff']:+.1f}{stars(r.p_perm)} | "
                             f"[{r.ci_low:+.1f}, {r.ci_high:+.1f}] | {r.p_perm:.3f} | {r.p_wilcoxon:.3f} |")
        lines.append("")
    return "\n".join(lines)

(ROOT / "results_table_crossimg.md").write_text(to_markdown(summary, METHOD_ORDER))
print("Guardado results_table_crossimg.md")

## Nota sobre Deep Learning

Los modelos de DL (U-Net, DeepLab, FCN) **ya son cross-image por naturaleza**: se entrena un
modelo por arquitectura y condición sobre un split de entrenamiento que mezcla orientaciones, y
se evalúa por set (los **6 modelos** = 3 arquitecturas × 2 condiciones). Esa sección está en el
notebook principal `Experimento_BoxCox_Segmentacion.ipynb` (sección 5).